In [11]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_validate,
    GridSearchCV,
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
)

from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
)

from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

DATA_PATH = "housing.csv"
TARGET_COLUMN = "median_house_value"


def load_data(path: str) -> pd.DataFrame:
    """Load housing dataset."""
    return pd.read_csv(path)


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """Create preprocessing pipeline."""

    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ])


def evaluate_model(model, X, y) -> dict:
    """Calculate regression metrics."""

    predictions = model.predict(X)

    return {
        "RMSE": root_mean_squared_error(y, predictions),
        "MAE": mean_absolute_error(y, predictions),
        "R2": r2_score(y, predictions),
    }


def compare_models(preprocessor, X_train, y_train):

    models = {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(random_state=RANDOM_STATE),
        "Lasso": Lasso(
            random_state=RANDOM_STATE,
            max_iter=10000,
        ),
        "RandomForest": RandomForestRegressor(
            random_state=RANDOM_STATE,
        ),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            random_state=RANDOM_STATE,
        ),
    }

    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    scoring = {
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2",
    }

    results = []

    for name, estimator in models.items():

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", estimator),
        ])

        scores = cross_validate(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
        )

        results.append({
            "Model": name,
            "CV_RMSE": -scores["test_rmse"].mean(),
            "CV_MAE": -scores["test_mae"].mean(),
            "CV_R2": scores["test_r2"].mean(),
        })

    return (
        pd.DataFrame(results)
        .sort_values("CV_RMSE")
        .reset_index(drop=True)
    )


def tune_hist_gradient_boosting(
    preprocessor,
    X_train,
    y_train,
):

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", HistGradientBoostingRegressor(
            random_state=RANDOM_STATE,
        )),
    ])

    param_grid = {
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [None, 3, 6],
        "model__max_leaf_nodes": [15, 31, 63],
        "model__min_samples_leaf": [20, 50, 100],
        "model__l2_regularization": [0.0, 0.1, 1.0],
    }

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
        verbose=1,
    )

    search.fit(X_train, y_train)

    return search


def predict_price(model, **features) -> float:
    """
    Predict house price from feature values.
    """

    sample = pd.DataFrame([features])

    return float(model.predict(sample)[0])


def main():

    # Load data
    df = load_data(DATA_PATH)

    X = df.drop(columns=TARGET_COLUMN)
    y = df[TARGET_COLUMN]

    X_train, X_test, Y_train, Y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
# bulid preprocessing
    preprocessor = build_preprocessor(X_train)

   # baseline model
    baseline=Pipeline([("preprocessor",preprocessor),("model",LinearRegression()) ])
    baseline.fit(X_train,Y_train)
    print("\n Baseline performance")
    print(evaluate_model(baseline,X_test,Y_test))
    # model comparsion

    print("\nCross Validation Results")
    cv_result=compare_models(preprocessor,X_train,Y_train)

    print(cv_result)
    #hyperparameter tuning
    search =tune_hist_gradient_boosting(preprocessor,X_train,Y_train)
    print("\nBest Parameters")
    print(search.best_params_)
    # final model 
    final_model=search.best_estimator_
    print("\n final test metrics")
    print(evaluate_model(final_model,X_test,Y_test))
if __name__ == "__main__":
    
    main()


 Baseline performance
{'RMSE': 18466.294732807564, 'MAE': 16255.326576577705, 'R2': 0.9908112688481215}

Cross Validation Results
                  Model       CV_RMSE        CV_MAE     CV_R2
0                 Lasso  17490.379333  14917.446100  0.992518
1      LinearRegression  17491.354984  14917.999489  0.992517
2                 Ridge  17507.837802  14909.024939  0.992506
3  HistGradientBoosting  23337.916508  19431.602911  0.986728
4          RandomForest  24676.655864  20511.605675  0.985149
Fitting 5 folds for each of 243 candidates, totalling 1215 fits

Best Parameters
{'model__l2_regularization': 1.0, 'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__max_leaf_nodes': 15, 'model__min_samples_leaf': 20}

 final test metrics
{'RMSE': 21802.664216199522, 'MAE': 18801.63433865123, 'R2': 0.9871910021816436}


In [1]:

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

dff=pd.DataFrame(pd.read_csv("housing1.csv"))
dff=pd.get_dummies(dff,columns=["location"],dtype=int)

scaler=StandardScaler()
scaler1=StandardScaler()

dff["sqft_living"] = scaler.fit_transform(dff[["sqft_living"]])
dff["price"] = scaler1.fit_transform(dff[["price"]])

dff["category"]=pd.qcut(dff["price"],q=3,labels=["Cheap","Moderate","Expensive"])

split=StratifiedShuffleSplit(n_splits=1,test_size=0.3)
for train_idx,test_idx in split.split(dff.drop (columns=["price"],axis=1).copy(),dff["category"]):
    train_df=dff.loc[train_idx]
    test_df=dff.loc[test_idx]
    
train_df=train_df.drop(columns='category')
test_df=test_df.drop(columns='category')
x_train=train_df.drop(columns="price").copy()
x_test=test_df.drop(columns="price").copy()

y_train=train_df['price']
y_test=test_df['price']

In [2]:
model=LinearRegression()
model.fit(x_train,y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [3]:
y_train=pd.Series(np.reshape(y_train,(-1)))
y_test=pd.Series(np.reshape(y_test,(-1)))

model.score(x_test,y_test)

0.9994871790564926

In [4]:
y_pred = model.predict(x_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :",mae)
print("RMSE:",rmse)
print("R2 Score:",r2)

MAE : 0.01922488317085822
RMSE: 0.022870228953408765
R2 Score: 0.9994871790564926


In [5]:
new_data = pd.DataFrame({
    "floors": [2],
    "sqft_living": [2500],
    "bedrooms": [3],
    "bathrooms": [2],
    "location": ["Noida"]
})
new_data = pd.get_dummies(new_data)
new_data = new_data.reindex(columns=x_train.columns, fill_value=0)
prediction = model.predict(new_data)
print("Predicted Price:", prediction[0])

Predicted Price: 2475.6659939245183
